<a href="https://colab.research.google.com/github/mahadikprasad15/ARENA/blob/main/Probes_for_finding_capitalization_feature.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install transformer_lens

In [ ]:
import torch
import torch.nn as nn
from transformer_lens import HookedTransformer
from typing import List, Dict, Tuple
import numpy as np
import random
from dataclasses import dataclass
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(42)
torch.manual_seed(42)

In [ ]:
model = HookedTransformer.from_pretrained('gpt2-small')
model.eval()

In [ ]:
@dataclass
class CapitalizationSample:

    prompt: str
    label: int  # 0 = not capitalized, 1 = capitalized
    target_word: str
    target_position: int  # token position to probe

def create_capitalization_dataset(
    n_samples: int = 1000,
    seed: int = 42
) -> Tuple[List[str], torch.Tensor, List[int]]:
    """
    Create balanced dataset for capitalization detection.

    Template: "The word [TARGET] is interesting"

    Returns:
        prompts: List of prompt strings
        labels: Tensor [n_samples] with 0/1
        target_positions: List of token positions (where TARGET is)

    Design decisions to think about:
    - How to ensure 50/50 balance?
    - Should you use the same words capitalized and uncapitalized?
    - How many unique words do you need?
    """
    random.seed(seed)

    words = [
        "apple", "river", "mountain", "coffee", "music",
        "ocean", "forest", "garden", "sunset", "window",
        "bridge", "castle", "desert", "island", "valley"
    ]


    prompt_template = 'This word {} is interesting'
    prompts_1 = [prompt_template.format(random.choice(words).capitalize()) for _ in range(int(n_samples / 2))]
    labels_1 = torch.ones(len(prompts_1), dtype = torch.long)
    prompts_0 = [prompt_template.format(random.choice(words)) for _ in range(int(n_samples / 2))]
    labels_0 = torch.zeros(len(prompts_1), dtype = torch.long)

    labels = torch.cat([labels_1, labels_0])
    prompts_1.extend(prompts_0)
    prompts = prompts_1

    target_positions = torch.cat([labels_1 * 3,labels_1 * 3]).tolist()

    return prompts,labels ,target_positions

In [ ]:
prompts, labels, positions = create_capitalization_dataset(100)
tokens = model.to_tokens(prompts)

positions_tensor = torch.tensor(positions)

dummy_activations = torch.randn(len(prompts), tokens.shape[1], 10)

selected_activations = dummy_activations[torch.arange(len(prompts)), positions_tensor]

print("Shape of selected activations:", selected_activations.shape)

In [ ]:
def cache_residual_activations(
    model: HookedTransformer,
    tokens: torch.Tensor,
    target_positions: List[int],
    layers: List[int] = None
) -> Dict[int, torch.Tensor]:

    layer_activations = {}
    if layers is None:
        layers = list(range(model.cfg.n_layers))

    target_positions = torch.tensor(target_positions)

    _, cache = model.run_with_cache(tokens)
    for layer in tqdm(layers):
      activation = cache[f'blocks.{layer}.hook_resid_post']

      activation = activation[torch.arange(tokens.size(0)), target_positions]
      layer_activations[f'blocks.{layer}.hook_resid_post'] = activation

    return layer_activations


In [ ]:
cache_dict = cache_residual_activations(model, tokens, positions)

In [ ]:
activations = {int(x.split('.')[1]): y for x,y in cache_dict.items()}

In [ ]:
class LinearProbe(nn.Module):
  def __init__(self, d_model, n_classes):
    super().__init__()

    self.linear = nn.Linear(d_model, n_classes)


  def forward(self, x):
    return self.linear(x)

In [ ]:
def split_data(
    activations_by_layer: Dict[int, torch.Tensor],
    labels: torch.Tensor,
    prompts: List[str],
    train_frac: float = 0.8,
    seed: int = 42
) -> Tuple[Dict, Dict, torch.Tensor, torch.Tensor, List[str], List[str]]:
    """
    Split activations, labels, and prompts into train/test sets.

    Returns:
        train_acts_by_layer: Dict[layer -> train_activations]
        test_acts_by_layer: Dict[layer -> test_activations]
        train_labels: Train labels
        test_labels: Test labels
        train_prompts: Train prompts (for later analysis)
        test_prompts: Test prompts

    CRITICAL: The split must be CONSISTENT across all layers!
    If sample 5 is in train set, it must be in train for ALL layers.

    STRATEGY:
    1. Generate random split indices once
    2. Apply same indices to all layers
    """


    # THINKING NUDGE 1: Random split
    # Create random permutation of indices
    # Split into train_indices and test_indices
    # Use torch.randperm()

    # THINKING NUDGE 2: Applying split
    # For each layer's activations, index with train_indices and test_indices
    # Same for labels and prompts

    # YOUR CODE HERE

    torch.manual_seed(seed)
    n_samples = labels.shape[0]
    split_point = int(train_frac * n_samples)


    shuffled_indices = torch.randperm(n_samples)
    train_indices = shuffled_indices[:split_point].tolist()

    test_indices = shuffled_indices[split_point:].tolist()


    train_acts_by_layer = {x: y[train_indices, :] for x,y in activations_by_layer.items()}
    test_acts_by_layer = {x: y[test_indices, :] for x,y in activations_by_layer.items()}
    train_labels = labels[train_indices]

    test_labels = labels[test_indices]

    train_prompts = [prompts[x] for x in train_indices]

    test_prompts = [prompts[x] for x in test_indices]

    return (train_acts_by_layer, test_acts_by_layer,
            train_labels, test_labels,
            train_prompts, test_prompts)

# Test
train_acts, test_acts, train_labels, test_labels, train_prompts, test_prompts = split_data(
    activations, labels[:20], prompts[:20], train_frac=0.8
)

assert train_labels.shape[0] == 16, f"Expected 16 train samples"
assert test_labels.shape[0] == 4, f"Expected 4 test samples"
assert len(train_prompts) == 16
assert all(train_acts[layer].shape[0] == 16 for layer in train_acts), "Inconsistent split!"

print(f"✓ Train size: {len(train_labels)}")
print(f"✓ Test size: {len(test_labels)}")
print(f"✓ Train label distribution: {train_labels.sum().item()}/{len(train_labels)}")

In [ ]:
def train_one_epoch(
    probe: nn.Module,
    train_activations: torch.Tensor,
    train_labels: torch.Tensor,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    batch_size: int = 64,
    device: str = "cpu"
) -> Tuple[float, float]:
    """
    Train probe for one epoch on cached activations.

    Returns:
        avg_loss: Average loss over epoch
        accuracy: Training accuracy

    THINKING:
    - Activations are already cached (not recomputing forward pass!)
    - Just training the probe weights
    - Need to create batches from cached activations
    """

    probe.train()
    train_activations = train_activations.to(device)
    train_labels = train_labels.to(device)

    n_samples = train_activations.shape[0]


    random_indices = torch.randperm(n_samples)
    batches = torch.split(random_indices,  batch_size)

    total_loss = 0

    for batch in batches:
      batch_acts = train_activations[batch]
      batch_labels = train_labels[batch]

      optimizer.zero_grad()
      logits = probe(batch_acts)
      loss = criterion(logits, batch_labels)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()
      accuracy =  0.0

    avg_loss = total_loss/ len(batches)

    return avg_loss, accuracy

# Test (mini test with small data)
test_probe = LinearProbe(768, 2).to(device)
test_optimizer = torch.optim.AdamW(test_probe.parameters(), lr=1e-3)
test_criterion = nn.CrossEntropyLoss()

loss, acc = train_one_epoch(
    test_probe,
    train_acts[0],  # Layer 0 activations
    train_labels,
    test_optimizer,
    test_criterion,
    batch_size=8,
    device=device
)

assert isinstance(loss, float), "Loss should be a float"
assert isinstance(acc, float), "Accuracy should be a float"
assert 0 <= acc <= 1, f"Accuracy should be in [0,1], got {acc}"

print(f"✓ Single epoch complete")
print(f"✓ Loss: {loss:.4f}")
print(f"✓ Accuracy: {acc:.4f}")

In [ ]:
total_size = activations[0].size(0)
random_indices = torch.randperm(total_size)
batches = torch.split(random_indices,  64)
batches

In [ ]:
labels